In [ ]:
# runner_verbose.py — paste into a notebook cell
import time, csv, os, sys, importlib
from datetime import datetime, timezone
from pathlib import Path
from tqdm import tqdm

# reload attacker module to ensure OR-Tools availability is refreshed
import anonlab.attacker_module as _am
importlib.reload(_am)

from anonlab import make_synthetic_patients, anonymize_qi, group_size_summary
from anonlab.attacker_module import make_attacker_subset_and_validate, run_attack

OUT_CSV = Path("sweep_results.csv")
LOGFILE = Path("sweep_run.log")

# ensure CSV header exists
def init_csv():
    header = [
        "timestamp","k","age_bin_width","topcode_start","rare_zip_min_frac",
        "allow_suppression","allow_zip_force","allow_sex_any",
        "attacker_frac","solver",
        "n_anon","suppressed","n_attack","n_candidates","avg_cands_per_attacker",
        "hits","hit_rate","time_anonymize_s","time_attack_s","total_time_s"
    ]
    if not OUT_CSV.exists():
        with OUT_CSV.open("w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow(header)

def log(msg):
    ts = datetime.now(timezone.utc).isoformat()
    with LOGFILE.open("a") as f:
        f.write(f"{ts} {msg}\n")
    print(msg)

def append_row(rowdict):
    # keep ordering consistent with header above
    row = [
        datetime.now(timezone.utc).isoformat(),
        rowdict.get("k"),
        rowdict.get("age_bin_width"),
        rowdict.get("topcode_start"),
        rowdict.get("rare_zip_min_frac"),
        rowdict.get("allow_suppression"),
        rowdict.get("allow_zip_force"),
        rowdict.get("allow_sex_any"),
        rowdict.get("attacker_frac"),
        rowdict.get("solver"),
        rowdict.get("n_anon"),
        rowdict.get("suppressed"),
        rowdict.get("n_attack"),
        rowdict.get("n_candidates"),
        rowdict.get("avg_cands_per_attacker"),
        rowdict.get("hits"),
        rowdict.get("hit_rate"),
        rowdict.get("time_anonymize_s"),
        rowdict.get("time_attack_s"),
        rowdict.get("total_time_s"),
    ]
    # append and flush immediately
    with OUT_CSV.open("a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(row)
        f.flush()


In [ ]:
# grid_runner_verbose.py
# from runner_verbose import init_csv, log, append_row
from tqdm import tqdm
from itertools import product
import time

init_csv()
df = make_synthetic_patients(n=2000, seed=421)

ks = [0,1,2,3,5,10]
age_bins = [0,1,2,5]
rare_fracs = [0.0, 0.01]

total = len(ks)*len(age_bins)*len(rare_fracs)
pbar = tqdm(total=total, desc="sweep", unit="run")

for k, age_w, rare in product(ks, age_bins, rare_fracs):
    t_start = time.time()
    anonymize_kwargs = dict(
        k=k,
        age_bin_width=age_w,
        topcode_start=75,
        rare_zip_min_frac=rare,
        max_iter=8,
        extra_iter=6,
        allow_zip_force=True,
        allow_sex_any=True,
        allow_suppression=True
    )
    attack_kwargs = dict(solver="greedy", weight_zip=1.0, weight_sex=0.8, age_scale=5.0)
    try:
        # anonymize + attack
        anon_df = anonymize_qi(df, **anonymize_kwargs)
        att = make_attacker_subset_and_validate(df, anon_df, fraction=0.1, seed=1)
        attacker_orig = att["attacker_orig"][["person_id","age","zip3","sex"]].reset_index(drop=True)
        attacker_ids = att["attacker_ids"]
        res = run_attack(attacker_orig, anon_df, attacker_ids, **attack_kwargs)
    except Exception as e:
        log(f"ERROR for k={k} bw={age_w} rare={rare}: {e}")
        # still record a failure row
        summary = {"k":k,"age_bin_width":age_w,"topcode_start":75,"rare_zip_min_frac":rare,
                   "allow_suppression":True,"allow_zip_force":True,"allow_sex_any":True,
                   "attacker_frac":0.1,"solver":attack_kwargs["solver"],
                   "n_anon": getattr(anon_df, "__len__", lambda: None)(),
                   "suppressed": getattr(anon_df, "attrs", {}).get("suppressed", None),
                   "n_attack": None,"n_candidates":None,"avg_cands_per_attacker":None,
                   "hits":None,"hit_rate":None,"time_anonymize_s":None,"time_attack_s":None,"total_time_s":None}
        append_row(summary)
        pbar.update(1)
        continue

    n_candidates = len(res.get("candidates", [])) if res.get("candidates") is not None else 0
    avg_cands = (n_candidates / res["eval"]["n_attack"]) if res["eval"]["n_attack"] else 0
    t_end = time.time()
    summary = {
        "k":k,"age_bin_width":age_w,"topcode_start":75,"rare_zip_min_frac":rare,
        "allow_suppression":True,"allow_zip_force":True,"allow_sex_any":True,
        "attacker_frac":0.1,"solver":res.get("solver_used", attack_kwargs["solver"]),
        "n_anon": len(anon_df),"suppressed": anon_df.attrs.get("suppressed",0),
        "n_attack": res["eval"]["n_attack"],"n_candidates": n_candidates,"avg_cands_per_attacker": round(avg_cands,2),
        "hits": res["eval"]["hits"],"hit_rate": round(res["eval"]["hit_rate"],4),
        "time_anonymize_s": round(0,3),"time_attack_s": round(0,3),"total_time_s": round(t_end-t_start,3)
    }
    # you can compute times granularly if desired; kept minimal here
    append_row(summary)
    log(f"done k={k} bw={age_w} rare={rare} => hit_rate={summary['hit_rate']:.3f} n_anon={summary['n_anon']} sup={summary['suppressed']}")
    pbar.update(1)

pbar.close()
log("SWEEP COMPLETE")


## plots

In [ ]:
# heatmaps.py
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("sweep_results.csv")
df = df.dropna(subset=["hit_rate", "k", "age_bin_width", "rare_zip_min_frac"])
df["k"] = df["k"].astype(int)
df["age_bin_width"] = df["age_bin_width"].astype(int)

for rare, g in df.groupby("rare_zip_min_frac"):
    pivot = g.pivot_table(index="k", columns="age_bin_width", values="hit_rate", aggfunc="mean")
    Z = pivot.values.astype(float)
    fig = plt.figure()
    im = plt.imshow(Z, aspect="auto", origin="lower", interpolation="nearest")
    plt.title(f"Hit@1 heatmap — rare_zip_min_frac={rare}")
    plt.xlabel("age_bin_width")
    plt.ylabel("k")
    plt.xticks(ticks=np.arange(len(pivot.columns)), labels=pivot.columns)
    plt.yticks(ticks=np.arange(len(pivot.index)), labels=pivot.index)
    cbar = plt.colorbar(im)
    cbar.set_label("Hit@1 rate")
    plt.tight_layout()
    plt.savefig(f"heatmap_hit_rate_rare_{rare}.png", dpi=160)
    plt.show()


In [ ]:
# lines_by_agebin.py
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("sweep_results.csv")
df = df.dropna(subset=["hit_rate","k","age_bin_width"])
df["k"] = df["k"].astype(int)
df["age_bin_width"] = df["age_bin_width"].astype(int)

for bw, g in df.groupby("age_bin_width"):
    g2 = g.groupby("k", as_index=False)["hit_rate"].mean().sort_values("k")
    fig = plt.figure()
    plt.plot(g2["k"], g2["hit_rate"], marker="o")
    plt.title(f"Hit@1 vs k — age_bin_width={bw}")
    plt.xlabel("k")
    plt.ylabel("Hit@1 rate")
    plt.grid(True, linestyle="--", linewidth=0.5)
    plt.tight_layout()
    plt.savefig(f"line_hit_rate_k_bw{bw}.png", dpi=160)
    plt.show()


In [ ]:
# scatter_candidates_vs_hit.py
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("sweep_results.csv")
df = df.dropna(subset=["hit_rate","avg_cands_per_attacker"])
x = df["avg_cands_per_attacker"].astype(float)
y = df["hit_rate"].astype(float)

fig = plt.figure()
plt.scatter(x, y, alpha=0.7)
plt.title("Hit@1 vs Avg candidates per attacker")
plt.xlabel("Avg candidates per attacker")
plt.ylabel("Hit@1 rate")
plt.grid(True, linestyle="--", linewidth=0.5)
plt.tight_layout()
plt.savefig("scatter_hit_vs_avgcands.png", dpi=160)
plt.show()


In [ ]:
# bars_suppression.py
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("sweep_results.csv")
df = df.dropna(subset=["k","suppressed","n_anon"])
df["k"] = df["k"].astype(int)

g = df.groupby("k", as_index=False).agg(
    suppressed_mean=("suppressed","mean"),
    n_anon_mean=("n_anon","mean")
)
fig = plt.figure()
plt.bar(g["k"], g["suppressed_mean"])
plt.title("Mean suppressed rows vs k (across grid)")
plt.xlabel("k")
plt.ylabel("Suppressed (mean)")
plt.tight_layout()
plt.savefig("bar_suppressed_vs_k.png", dpi=160)
plt.show()


In [ ]:
# compare_solver.py
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("sweep_results.csv")
if "solver" in df.columns:
    df = df.dropna(subset=["hit_rate","solver","k"])
    for s, g in df.groupby("solver"):
        g2 = g.groupby("k", as_index=False)["hit_rate"].mean().sort_values("k")
        fig = plt.figure()
        plt.plot(g2["k"], g2["hit_rate"], marker="o")
        plt.title(f"Solver comparison: {s} — Hit@1 vs k")
        plt.xlabel("k"); plt.ylabel("Hit@1 rate"); plt.grid(True, linestyle="--", linewidth=0.5)
        plt.tight_layout()
        plt.savefig(f"solver_{s}_hit_vs_k.png", dpi=160)
        plt.show()


## experiment 2

In [ ]:
# sweep_multi_seed.py
from datetime import datetime, timezone
from itertools import product
import time, csv, importlib
from pathlib import Path
from tqdm import tqdm

import anonlab.attacker_module as _am
importlib.reload(_am)

from anonlab import make_synthetic_patients, anonymize_qi
from anonlab.attacker_module import make_attacker_subset_and_validate, run_attack

OUT = Path("sweep_results_multi.csv")
HEAD = ["timestamp","seed","k","age_bin_width","rare_zip_min_frac","topcode_start",
        "allow_suppression","allow_zip_force","allow_sex_any",
        "attacker_frac","solver","n_anon","suppressed","n_attack",
        "n_candidates","avg_cands_per_attacker","hits","hit_rate",
        "time_anonymize_s","time_attack_s","total_time_s"]

def ts(): return datetime.now(timezone.utc).isoformat()

def ensure_header():
    if not OUT.exists():
        with OUT.open("w", newline="") as f:
            csv.writer(f).writerow(HEAD)

def write_row(d):
    row = [d.get(h) for h in HEAD]
    with OUT.open("a", newline="") as f:
        w = csv.writer(f); w.writerow(row); f.flush()

# ------- grid (adjust here) -------
seeds = [1,2,3,4,5,6,7]                   # more trials → smoother plots
ks = [1,2,3,4,5,6,7,8,9,10]      # k=0 omitted
age_bins = [1,2,3,4,5,6,7,8,9,10]         # 0 means width→1 via anonymizer
rare_fracs = [0] #= [0.0, 0.01, 0.02, 0.05]
topcode = 75
attacker_frac = 0.1
solver = "ortools" if getattr(_am, "ORTOOLS_AVAILABLE", False) else "greedy"
# ----------------------------------
patients = 200
ensure_header()
total = len(seeds)*len(ks)*len(age_bins)*len(rare_fracs)
pbar = tqdm(total=total, desc="sweep(multi)", unit="run")

for seed, k, bw, r in product(seeds, ks, age_bins, rare_fracs):
    df = make_synthetic_patients(n=patients, seed=421)  # fixed base population; vary attacker split via seed
    anon_kwargs = dict(
        k=k, age_bin_width=bw, topcode_start=topcode, rare_zip_min_frac=r,
        max_iter=8, extra_iter=6, allow_zip_force=True, allow_sex_any=True, allow_suppression=True
    )
    attack_kwargs = dict(solver=solver, weight_zip=1.0, weight_sex=0.8, age_scale=5.0)
    try:
        t0 = time.time()
        anon_df = anonymize_qi(df, **anon_kwargs); t1 = time.time()
        att = make_attacker_subset_and_validate(df, anon_df, fraction=attacker_frac, seed=seed)
        attacker_orig = att["attacker_orig"][["person_id","age","zip3","sex"]].reset_index(drop=True)
        attacker_ids = att["attacker_ids"]
        res = run_attack(attacker_orig, anon_df, attacker_ids, **attack_kwargs); t2 = time.time()

        n_cands = len(res.get("candidates", [])) if res.get("candidates") is not None else 0
        avg_cands = (n_cands / res["eval"]["n_attack"]) if res["eval"]["n_attack"] else 0.0

        write_row(dict(
            timestamp=ts(), seed=seed, k=k, age_bin_width=bw, rare_zip_min_frac=r, topcode_start=topcode,
            allow_suppression=True, allow_zip_force=True, allow_sex_any=True,
            attacker_frac=attacker_frac, solver=res.get("solver_used", solver),
            n_anon=len(anon_df), suppressed=anon_df.attrs.get("suppressed",0),
            n_attack=res["eval"]["n_attack"], n_candidates=n_cands, avg_cands_per_attacker=round(avg_cands,2),
            hits=res["eval"]["hits"], hit_rate=round(res["eval"]["hit_rate"],6),
            time_anonymize_s=round(t1-t0,3), time_attack_s=round(t2-t1,3), total_time_s=round(t2-t0,3)
        ))
    except Exception as e:
        # still record failure row (helps you spot bad regions)
        write_row(dict(
            timestamp=ts(), seed=seed, k=k, age_bin_width=bw, rare_zip_min_frac=r, topcode_start=topcode,
            allow_suppression=True, allow_zip_force=True, allow_sex_any=True,
            attacker_frac=attacker_frac, solver=solver,
            n_anon=None, suppressed=None, n_attack=None, n_candidates=None, avg_cands_per_attacker=None,
            hits=None, hit_rate=None, time_anonymize_s=None, time_attack_s=None, total_time_s=None
        ))
    pbar.update(1)

pbar.close()
print("DONE ->", OUT)


In [ ]:
# plots_heatmaps_mean.py
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("sweep_results_multi.csv")
df = df.dropna(subset=["hit_rate","k","age_bin_width","rare_zip_min_frac"])
df["k"] = df["k"].astype(int)
df = df[df["k"] >= 1]  # drop k=0 if any
df["age_bin_width"] = df["age_bin_width"].astype(int)

for rare, g in df.groupby("rare_zip_min_frac"):
    gm = g.groupby(["k","age_bin_width"], as_index=False)["hit_rate"].mean()
    ks = sorted(gm["k"].unique())
    bws = sorted(gm["age_bin_width"].unique())
    M = np.full((len(ks), len(bws)), np.nan)
    for i,k in enumerate(ks):
        row = gm[gm["k"]==k]
        vals = row.set_index("age_bin_width")["hit_rate"].reindex(bws)
        M[i,:] = vals.values

    fig = plt.figure(figsize=(7,5))
    # im = plt.imshow(M, aspect="auto", origin="lower", interpolation="nearest")
    im = plt.imshow(
            M,
            aspect="auto",
            origin="lower",
            interpolation="nearest",
            cmap=plt.cm.copper  # ← copper palette
        )

    plt.title(f"Hit@1 (mean across seeds) — rare_zip_min_frac={rare}")
    plt.xlabel("age_bin_width")
    plt.ylabel("k")
    plt.xticks(ticks=np.arange(len(bws)), labels=bws)
    plt.yticks(ticks=np.arange(len(ks)),  labels=ks)
    cbar = plt.colorbar(im); cbar.set_label("Hit@1")
    plt.tight_layout()
    plt.savefig(f"heatmap_mean_hit_rate_rare_{rare}.png", dpi=220)
    plt.show()


In [ ]:
# plots_tradeoff.py
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("sweep_results_multi.csv").dropna(subset=["hit_rate","n_anon","suppressed","k"])
df = df[df["k"] >= 1]
# crude utility proxy: released fraction; feel free to replace with your true utility metric later
df["released_frac"] = df["n_anon"] / (df["n_anon"] + df["suppressed"].fillna(0))
g = df.groupby(["k","age_bin_width"], as_index=False)[["hit_rate","released_frac"]].mean()

fig = plt.figure(figsize=(6,5))
plt.scatter(g["released_frac"], g["hit_rate"], alpha=0.7)
plt.title("Tradeoff: Released fraction vs Hit@1 (means over seeds)")
plt.xlabel("Released fraction")
plt.ylabel("Hit@1")
plt.grid(True, linestyle="--", linewidth=0.6)
plt.tight_layout()
plt.savefig("scatter_tradeoff_release_vs_hit.png", dpi=220)
plt.show()


In [ ]:
# plots_candidates_pressure.py
import pandas as pd, numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("sweep_results_multi.csv")
df = df.dropna(subset=["hit_rate","avg_cands_per_attacker","k"])
df = df[df["k"] >= 1]

agg = df.groupby(["k","age_bin_width","rare_zip_min_frac"], as_index=False).agg(
    hit_mean=("hit_rate","mean"),
    cands_mean=("avg_cands_per_attacker","mean")
)
fig = plt.figure(figsize=(6,5))
plt.scatter(agg["cands_mean"], agg["hit_mean"], alpha=0.7)
plt.title("Hit@1 vs Avg candidates per attacker (cell means)")
plt.xlabel("Avg candidates per attacker")
plt.ylabel("Hit@1")
plt.grid(True, linestyle="--", linewidth=0.6)
plt.tight_layout()
plt.savefig("scatter_hit_vs_candidates_mean.png", dpi=220)
plt.show()
